# 1. Config e imports

In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 31.8 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import f1_score

# Configuración de dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizando dispositivo: {device}")

# Rutas de entrada (Asegúrate de haber añadido el dataset con los folds de la Fase 1)
TRAIN_FOLDS_PATH = "/kaggle/input/datasets/ernestomiguez/casmi2026-phase1-folds/train_folds.parquet"
TEST_PATH = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet"
TRAIN_PATH = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet"

Utilizando dispositivo: cuda


# 2. Funciones de Limpieza de Espectros MS2

In [3]:
def clean_spectrum(mzs, intensities, precursor_mz, top_n=128, min_rel_intensity=0.01):
    """
    Limpia y filtra un espectro MS2 según las reglas recomendadas de la competición:
    1. Elimina picos > precursor_mz + 2.0 Da.
    2. Elimina picos con intensidad relativa por debajo del min_rel_intensity (1%).
    3. Mantiene los top_n picos más intensos.
    4. Aplica transformación de raíz cuadrada a las intensidades.
    """
    mzs = np.array(mzs, dtype=np.float32)
    intensities = np.array(intensities, dtype=np.float32)
    
    # 1. Filtro de masa superior al precursor
    mask_mass = mzs <= (precursor_mz + 2.0)
    mzs, intensities = mzs[mask_mass], intensities[mask_mass]
    
    if len(mzs) == 0:
        return np.array([0.0], dtype=np.float32), np.array([0.0], dtype=np.float32)
    
    # 2. Re-normalizar al pico base y filtrar por intensidad relativa mínima
    max_intensity = intensities.max()
    if max_intensity > 0:
        rel_intensities = intensities / max_intensity
    else:
        rel_intensities = intensities
        
    mask_int = rel_intensities >= min_rel_intensity
    mzs, intensities = mzs[mask_int], rel_intensities[mask_int]
    
    # 3. Retener Top-N picos
    if len(mzs) > top_n:
        top_indices = np.argsort(intensities)[-top_n:]
        mzs, intensities = mzs[top_indices], intensities[top_indices]
        # Reordenar por m/z ascendente
        sort_mz_idx = np.argsort(mzs)
        mzs, intensities = mzs[sort_mz_idx], intensities[sort_mz_idx]
        
    # 4. Transformación Square-Root para suavizar picos dominantes
    intensities = np.sqrt(intensities)
    
    return mzs, intensities

def spectrum_to_binned_vector(mzs, intensities, bin_size=0.1, max_mz=1200.0):
    """
    Convierte la lista de picos (m/z, intensidad) en un vector denso por bining.
    """
    num_bins = int(max_mz / bin_size)
    vector = np.zeros(num_bins, dtype=np.float32)
    
    indices = (mzs / bin_size).astype(int)
    valid_mask = (indices >= 0) & (indices < num_bins)
    
    for idx, int_val in zip(indices[valid_mask], intensities[valid_mask]):
        vector[idx] = max(vector[idx], int_val)  # O acumulación / max
        
    return vector

# 3. Generación de Fingerprints (Targets) a partir de SMILES

In [4]:
# Instanciamos el generador de Morgan Fingerprints fuera para reutilizarlo
# radius=2 (equivalente a ECFP4), fpSize=1024 bits
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_morgan_fingerprint(smiles, n_bits=1024):
    """
    Genera un Morgan Fingerprint moderno (ECFP4) utilizando rdFingerprintGenerator.
    """
    if not isinstance(smiles, str) or not smiles.strip():
        return np.zeros(n_bits, dtype=np.float32)
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(n_bits, dtype=np.float32)
        
        # Generar fingerprint con la API moderna de RDKit
        fp = morgan_gen.GetFingerprint(mol)
        arr = np.zeros((n_bits,), dtype=np.uint8)
        Chem.DataStructs.ConvertToNumpyArray(fp, arr)
        return arr.astype(np.float32)
    except Exception:
        return np.zeros(n_bits, dtype=np.float32)

# 4. Dataset y Modelo PyTorch para Predicción de Fingerprints

In [5]:
class MS2Dataset(Dataset):
    def __init__(self, df, bin_size=0.5, max_mz=1000.0, is_train=True):
        self.df = df.reset_index(drop=True)
        self.bin_size = bin_size
        self.max_mz = max_mz
        self.num_bins = int(max_mz / bin_size)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Limpieza de espectro
        clean_mzs, clean_ints = clean_spectrum(
            row['ms2_mzs'], 
            row['ms2_normalized_intensities'], 
            row['precursor_mz']
        )
        
        # Binned vector
        spec_vec = spectrum_to_binned_vector(clean_mzs, clean_ints, self.bin_size, self.max_mz)
        
        # Añadir precursor_mz como feature escalar al final del vector
        x = np.append(spec_vec, np.float32(row['precursor_mz'] / 1000.0))
        
        if self.is_train:
            y = smiles_to_morgan_fingerprint(row['normalized_smiles'], n_bits=1024)
            return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)
        else:
            return torch.tensor(x, dtype=torch.float32)

class SpectrumToFingerprintMLP(nn.Module):
    def __init__(self, input_dim, output_dim=1024):
        super(SpectrumToFingerprintMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, output_dim),
            nn.Sigmoid()  # Salida de probabilidades para multilabel
        )

    def forward(self, x):
        return self.net(x)

# 5. Carga de Folds y Preparación de DataLoaders

In [6]:
print("Cargando dataset original y folds asignados...")
full_train_df = pd.read_parquet(TRAIN_PATH)
folds_df = pd.read_parquet(TRAIN_FOLDS_PATH)

if 'fold' in full_train_df.columns:
    full_train_df = full_train_df.drop(columns=['fold'])

train_folds_df = full_train_df.merge(
    folds_df[['normalized_smiles', 'inchikey14', 'fold']].drop_duplicates(),
    on=['normalized_smiles', 'inchikey14'],
    how='inner'
)

# Separar en Train y Validation usando el Fold 0
train_df = train_folds_df[train_folds_df['fold'] != 0].reset_index(drop=True)
val_df = train_folds_df[train_folds_df['fold'] == 0].reset_index(drop=True)

# Liberar memoria de los DataFrames pesados que ya no se usan
del full_train_df, train_folds_df
import gc; gc.collect()

print(f"Espectros para Entrenamiento: {len(train_df):,}")
print(f"Espectros para Validación:    {len(val_df):,}")

# Crear PyTorch Datasets
train_dataset = MS2Dataset(train_df, bin_size=0.5, max_mz=1000.0, is_train=True)
val_dataset = MS2Dataset(val_df, bin_size=0.5, max_mz=1000.0, is_train=True)

# DataLoaders optimizados para evitar RAM OOM
train_loader = DataLoader(
    train_dataset, 
    batch_size=256,        # Mantener en 256 para prevenir GPU VRAM OOM
    shuffle=True, 
    num_workers=2,         # Máximo 2 workers para la RAM de Kaggle
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=256, 
    shuffle=False, 
    num_workers=2, 
    pin_memory=True
)

print("✓ DataLoaders configurados para prevenir OOM.")

Cargando dataset original y folds asignados...
Espectros para Entrenamiento: 2,031,686
Espectros para Validación:    507,922
✓ DataLoaders configurados para prevenir OOM.


# 6. Instanciación del Modelo, Función de Pérdida y Optimizador

In [7]:
# Dimensión de entrada: num_bins (1000.0 / 0.5 = 2000) + 1 (precursor_mz)
INPUT_DIM = int(1000.0 / 0.5) + 1
OUTPUT_DIM = 1024  # Tamaño del Morgan Fingerprint

model = SpectrumToFingerprintMLP(input_dim=INPUT_DIM, output_dim=OUTPUT_DIM).to(device)

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, cosine_weight=0.5):
        super().__init__()
        self.bce = nn.BCELoss()
        self.bce_weight = bce_weight
        self.cosine_weight = cosine_weight
        
    def forward(self, preds, targets):
        # 1. Pérdida BCE tradicional (bit a bit)
        loss_bce = self.bce(preds, targets)
        
        # 2. Pérdida por Distancia Coseno (alineación global de vectores)
        # Cosine similarity entre predicciones y targets reales
        dot_product = (preds * targets).sum(dim=1)
        pred_norm = preds.norm(dim=1, p=2)
        target_norm = targets.norm(dim=1, p=2)
        
        cosine_sim = dot_product / (pred_norm * target_norm + 1e-8)
        loss_cosine = (1.0 - cosine_sim).mean()
        
        # Pérdida total combinada
        total_loss = (self.bce_weight * loss_bce) + (self.cosine_weight * loss_cosine)
        return total_loss

# Inicializar la nueva función de pérdida
criterion = CombinedLoss(bce_weight=0.5, cosine_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(model)

SpectrumToFingerprintMLP(
  (net): Sequential(
    (0): Linear(in_features=2001, out_features=1024, bias=True)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=1024, out_features=1024, bias=True)
    (5): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=1024, out_features=1024, bias=True)
    (9): Sigmoid()
  )
)


# 7. Bucle de entrenamiento

In [8]:
EPOCHS = 15
best_val_loss = float('inf')

# Programador de Tasa de Aprendizaje (Reduce el LR si val_loss no mejora en 2 épocas)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

for epoch in range(1, EPOCHS + 1):
    # --- ENTRENAMIENTO ---
    model.train()
    train_loss = 0.0
    
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * len(x_batch)
        
    train_loss /= len(train_dataset)
    
    # --- VALIDACIÓN ---
    model.eval()
    val_loss = 0.0
    val_preds_list = []
    val_targets_list = []
    
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            preds = model(x_batch)
            loss = criterion(preds, y_batch)
            val_loss += loss.item() * len(x_batch)
            
            # Almacenar predicciones y targets para métricas
            val_preds_list.append(preds.cpu())
            val_targets_list.append(y_batch.cpu())
            
    val_loss /= len(val_dataset)
    
    # --- CÁLCULO DE MÉTRICA QUÍMICA (F1-Score Micro) ---
    val_preds_arr = torch.cat(val_preds_list, dim=0).numpy()
    val_targets_arr = torch.cat(val_targets_list, dim=0).numpy()
    
    # Binarizar las predicciones con umbral 0.5
    val_preds_binary = (val_preds_arr > 0.5).astype(int)
    val_f1_micro = f1_score(val_targets_arr, val_preds_binary, average='micro')
    
    # Ajuste de Learning Rate según la pérdida de validación
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1 (Micro): {val_f1_micro:.4f} | LR: {current_lr:.1e}")
    
    # Guardar los mejores pesos basados en Val Loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'fingerprint_mlp_fold0.pth')
        print(f"  --> Guardado nuevo mejor modelo (Val Loss: {val_loss:.4f})")

print("\n¡Entrenamiento finalizado exitosamente!")

Epoch 01/15 | Train Loss: 0.2961 | Val Loss: 0.2959 | Val F1 (Micro): 0.4063 | LR: 1.0e-03
  --> Guardado nuevo mejor modelo (Val Loss: 0.2959)
Epoch 02/15 | Train Loss: 0.2823 | Val Loss: 0.2917 | Val F1 (Micro): 0.4081 | LR: 1.0e-03
  --> Guardado nuevo mejor modelo (Val Loss: 0.2917)
Epoch 03/15 | Train Loss: 0.2771 | Val Loss: 0.2901 | Val F1 (Micro): 0.4117 | LR: 1.0e-03
  --> Guardado nuevo mejor modelo (Val Loss: 0.2901)
Epoch 04/15 | Train Loss: 0.2740 | Val Loss: 0.2895 | Val F1 (Micro): 0.4126 | LR: 1.0e-03
  --> Guardado nuevo mejor modelo (Val Loss: 0.2895)
Epoch 05/15 | Train Loss: 0.2719 | Val Loss: 0.2890 | Val F1 (Micro): 0.4125 | LR: 1.0e-03
  --> Guardado nuevo mejor modelo (Val Loss: 0.2890)
Epoch 06/15 | Train Loss: 0.2703 | Val Loss: 0.2884 | Val F1 (Micro): 0.4139 | LR: 1.0e-03
  --> Guardado nuevo mejor modelo (Val Loss: 0.2884)
Epoch 07/15 | Train Loss: 0.2691 | Val Loss: 0.2882 | Val F1 (Micro): 0.4163 | LR: 1.0e-03
  --> Guardado nuevo mejor modelo (Val Loss: 

# 8. Guardar los Embeddings / Fingerprints predichos

In [9]:
# Cargar los mejores pesos guardados
model.load_state_dict(torch.load('fingerprint_mlp_fold0.pth'))
model.eval()

# Cargar el test.parquet
test_df = pd.read_parquet(TEST_PATH)
test_dataset = MS2Dataset(test_df, bin_size=0.5, max_mz=1000.0, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

test_preds = []
with torch.no_grad():
    for x_batch in test_loader:
        x_batch = x_batch.to(device)
        preds = model(x_batch)
        test_preds.append(preds.cpu().numpy())

test_preds_arr = np.concatenate(test_preds, axis=0)

# Guardar las predicciones de fingerprints para usarlas en la Fase 3
np.save('test_predicted_fingerprints.npy', test_preds_arr)
print("✓ Se han guardado 'fingerprint_mlp_fold0.pth' y 'test_predicted_fingerprints.npy'.")

✓ Se han guardado 'fingerprint_mlp_fold0.pth' y 'test_predicted_fingerprints.npy'.
